# 基于 MatmulLeakyRelu 样例开发 CV 融合算子

本节以 MatmulLeakyRelu 样例为主轴，重点分析 Matmul 计算结果如何搬运到 Vector 核、LeakyRelu 计算如何嵌入 Matmul 迭代，以及中间分片如何在 LocalTensor 中复用。

本节学习大纲：

- 理解 PyAsc 对 AIC→AIV CV 融合数据流的表达。
- 使用 `cat` 获取并阅读完整 MatmulLeakyRelu 样例。
- 识别 Matmul 计算、LeakyRelu 计算和 CopyOut 流程。
- 掌握 `VECCALC`、`iterate`、`get_tensor_c` 与 `leaky_relu` 的配合。
- 管理融合分片的 LocalTensor 生命周期并验证结果。

---

## 1. PyAsc 中的 CV 融合表达

PyAsc 使用 Matmul 高阶 API 表达 AIC 上的 Cube 计算，使用 Vector API 表达 AIV 上的后处理。MatmulLeakyRelu 算子将两部分放入同一个 Kernel：

```text
A / B / Bias（GM）
    -> Matmul（AIC）
    -> base_m * base_n 输出分片
    -> LeakyReLU（AIV）
    -> C（GM）
```

`matmul.iterate()` 产生矩阵分片，`matmul.get_tensor_c()` 将 Matmul 结果获取到 Vector 侧 LocalTensor，`asc.leaky_relu()` 在该 LocalTensor 上完成 Vector 计算，`asc.data_copy()` 再执行 CopyOut。Matmul 计算、LeakyRelu 计算和 CopyOut 都位于一个 Kernel 中，构成样例的 CV 融合流程。

<img src="./images/matmul-leakyrelu-pipeline.png" alt="MatmulLeakyRelu CV 融合流程" width="700px">

*图 5-3 MatmulLeakyRelu 的 CV 融合流程*

## 2. 获取样例并定位融合边界

MatmulLeakyRelu 的数学语义为：

```text
D = A * B + Bias
C = D >= 0 ? D : D * alpha
```

其中 Matmul 在 AIC 上进行计算，LeakyReLU 在 AIV 上进行计算，`D` 是需要从 Cube 核搬运到 Vector 核的 Matmul 中间结果。样例规格如下：

| 参数 | 输入/输出 | Shape | 数据类型 | 格式 |
| --- | --- | --- | --- | --- |
| `a` | 输入 | `[1024, 256]` | float16 | ND |
| `b` | 输入 | `[256, 640]` | float16 | ND |
| `bias` | 输入 | `[1, 640]` | float32 | ND |
| `alpha` | 输入标量 | `[]` | float32 | - |
| `c` | 输出 | `[1024, 640]` | float32 | ND |

完整样例源码位于 `src/05.03/matmul_leakyrelu.py`。执行下面的 `cat` 命令查看源码。

In [ ]:
!cat ./src/05.03/matmul_leakyrelu.py

## 3. 从完整源码中识别融合相关代码

阅读完整源码时，可以按下面四个角色定位 CV 融合逻辑：

| 角色 | 样例代码 | 作用 |
| --- | --- | --- |
| Matmul 对象初始化 | `asc.adv.Matmul`、`register_matmul` | 初始化 Matmul 对象 |
| Matmul 计算 | `iterate`、`get_tensor_c` | 逐块计算并获取 Matmul 中间结果 |
| LeakyRelu 计算 | `asc.leaky_relu`、`TQue.enque` | 对 Matmul 中间结果执行 Vector 计算并将结果入队 |
| CopyOut | `TQue.deque`、`data_copy`、`free_tensor` | 取出结果、搬运到 GM 并释放缓存 |

PyAsc 还提供 `asc.add_relu`、`asc.fused_mul_add` 和 `asc.fused_mul_add_relu` 等组合 Vector API。当后处理公式匹配时，可以用对应接口替换 AIV 上的 Vector 阶段；本样例的公式对应 `asc.leaky_relu`。

## 4. 关键点一：将 Matmul 结果用于 Vector 计算

Kernel 创建 Matmul 对象时，将 C 的逻辑位置设为 `VECCALC`；Host 生成 Tiling 时也对 C 使用相同位置：

```python
# Kernel 侧
c=asc.adv.MatmulType(
    asc.TPosition.VECCALC, asc.CubeFormat.ND, c_global.dtype
)

# Host Tiling 侧
matmul_tiling.set_c_type(
    host.TPosition.VECCALC, host.CubeFormat.ND, host.DataType.DT_FLOAT
)
```

这两处配置共同描述 Matmul 输出将继续参与 Vector 计算。Kernel 与 Host Tiling 必须对 C 的位置保持一致；如果更换输出位置，获取分片的方式也需要同步调整。

## 5. 关键点二：把 Vector 计算放入 Matmul 迭代

样例在每次 Matmul 迭代产生结果后立即执行 LeakyReLU，而不是等待完整矩阵计算结束：

```python
with matmul.iterate() as count:
    relu_out_local = relu_out_queue.alloc_tensor(c.dtype)
    matmul.get_tensor_c(relu_out_local, en_sequential_write=True)
    asc.leaky_relu(
        relu_out_local, relu_out_local, alpha,
        count=tiling.base_m * tiling.base_n,
    )
    relu_out_queue.enque(relu_out_local)
    relu_out_local = relu_out_queue.deque(c.dtype)
    # 样例计算写回偏移和搬运参数后，将当前分片写回 C
    asc.data_copy(c_global[start_offset:], relu_out_local, repeat_params=params)
    relu_out_queue.free_tensor(relu_out_local)
matmul.end()
```

这里有三个直接决定融合行为的调用：

1. `iterate` 每轮产生一个 `base_m * base_n` 的 Matmul 输出块。
2. `get_tensor_c` 将 Matmul 结果获取到 Vector 侧 LocalTensor。
3. `leaky_relu` 在同一 LocalTensor 上原地处理整个输出块。

`matmul.end()` 在全部分片处理完成后结束 Matmul 并释放其计算资源。

## 6. 融合分片的缓冲与写回

样例为一个 `base_m * base_n` 输出块分配 TQue 缓冲区。`num=1` 表示使用一个内存块；LeakyReLU 原地覆盖 Matmul 结果，因此不需要再申请一块同尺寸 LocalTensor 保存激活输出。

当前分片的生命周期为：

```text
alloc -> get_tensor_c -> leaky_relu -> enque -> deque -> data_copy -> free
```

`enque` 和 `deque` 建立 Vector 计算与写回之间的数据依赖。`data_copy` 完成后才能调用 `free_tensor`，随后同一队列内存可供下一次迭代复用。

LocalTensor 中的分片连续存放，而最终 C 是完整二维矩阵。样例使用 `start_offset` 定位当前输出块，并用 `DataCopyParams` 按行写回。具体偏移公式属于输出布局处理，不改变 CV 融合的核心交接顺序。

## 7. Host 配置与融合结果验证

Host 侧需要保证 Tiling 与 Kernel 的融合约定一致：C 的位置为 `VECCALC`，`base_m * base_n` 分片能够放入 Vector 侧缓冲区，Bias、数据类型和 Shape 与 Kernel 声明一致。其他多核遍历与矩阵分块参数继续服务于 Matmul 计算本身。

<img src="./images/matmul-fusion-workflow.png" alt="MatmulLeakyRelu Host 与 Kernel 流程" width="700px">

*图 5-4 MatmulLeakyRelu 的 Host 与 Kernel 流程*

验证时使用不依赖 Kernel 分片调度的表达式计算期望结果：

```python
matmul = torch.matmul(a.to(torch.float32), b.to(torch.float32)) + bias
expected = torch.where(matmul >= 0, matmul, matmul * alpha)
assert torch.allclose(actual, expected, rtol=1e-3, atol=1e-3)
```

这项比较同时检查融合后的数学语义和最终输出布局。断言只表示当前输入、Shape、数据类型与 Tiling 配置下的结果满足给定容差。

---

## 课后练习

### 第 1 题（单选）

为什么样例将 Matmul 的 C 类型配置到 `VECCALC`？

A. 让输入 A 自动转置。  
B. 让当前矩阵分片能够继续参与 Vector 计算。  
C. 跳过 Bias 计算。  
D. 将输出类型改为 float16。

### 第 2 题（单选）

MatmulLeakyRelu 融合主循环的正确顺序是什么？

A. `get_tensor_c -> iterate -> data_copy -> leaky_relu`  
B. `iterate -> get_tensor_c -> leaky_relu -> data_copy`  
C. `leaky_relu -> data_copy -> iterate -> get_tensor_c`  
D. `data_copy -> leaky_relu -> get_tensor_c -> iterate`

### 第 3 题（多选）

关于样例的 LocalTensor 生命周期，哪些说法正确？

A. 一个缓冲区容纳一个 `base_m * base_n` 输出分片。  
B. LeakyReLU 在同一个 LocalTensor 上原地计算。  
C. 分片写回后调用 `free_tensor`，缓冲区可供下一轮复用。  
D. 应在 `get_tensor_c` 前释放当前 LocalTensor。

### 第 4 题（多选）

样例如何验证融合结果？

A. 使用 `torch.matmul + bias` 构造矩阵计算。  
B. 使用 `torch.where` 表达 LeakyReLU。  
C. 使用带 `rtol` 和 `atol` 的 `torch.allclose` 比较结果。  
D. 只检查 Kernel 是否启动，不比较输出。

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/05.03_answer.txt